# Tier 2. Module 9 - Product Analytics and Applied Statistics

## Lesson 11. Homework - Cause and effect relationships in product analytics

### Technical Task

A company that develops a mobile language learning app has launched a new personalized recommendation feature. It is designed to help users better select learning materials and increase their engagement.

The product team wants to determine whether this feature is actually improving user retention, or whether the observed changes are due to other factors.

#### Task description

Upload a [data table](https://drive.google.com/file/d/1mxIREGZ7_RtH__6rKLnigmHbO5DI_Zne/view?usp=sharing) where:
- **Group**: “Test” – users who received the new feature, “Control” – those who did not.
- **Retention_7d**: Whether the user returned 7 days after registration (1 = yes, 0 = no).
- **Retention_30d**: Whether the user returned 30 days after registration.
- **Avg_Session_Time**: Average session time of the user.
- **Region**: Geographic region of the user.

In [2]:
import pandas as pd
from scipy.stats import ttest_ind, pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv("user_data_2000.csv")
df.head()

,User_ID,Group,Retention_7d,Retention_30d,Avg_Session_Time,Region
0,101,Test,0,1,7,EU
1,102,Control,1,0,13,US
2,103,Test,1,0,9,US
3,104,Test,1,0,12,EU
4,105,Test,1,1,11,EU


#### Task #1. Determine the correlation between feature usage and retention

Calculate the correlation coefficient between the usage of the new feature (Test Group) and the retention indicators (Retention_7d, Retention_30d).

Can we conclude that there is a causal relationship based on this?

---

Binary variable for test group

In [4]:
df['Is_Test'] = df['Group'].apply(lambda x: 1 if x == 'Test' else 0)
df.head()

,User_ID,Group,Retention_7d,Retention_30d,Avg_Session_Time,Region,Is_Test
0,101,Test,0,1,7,EU,1
1,102,Control,1,0,13,US,0
2,103,Test,1,0,9,US,1
3,104,Test,1,0,12,EU,1
4,105,Test,1,1,11,EU,1


Correlation calculation

In [5]:
corr_7d, p_7d = pearsonr(df['Is_Test'], df['Retention_7d'])
corr_30d, p_30d = pearsonr(df['Is_Test'], df['Retention_30d'])

print(f"Correlation with Retention_7d: {corr_7d:.3f} (p = {p_7d:.3f})")
print(f"Correlation with Retention_30d: {corr_30d:.3f} (p = {p_30d:.3f})")

Correlation with Retention_7d: -0.011 (p = 0.611)
Correlation with Retention_30d: -0.047 (p = 0.037)


#### Conclusion:

From this calculation, we can conclude that there no correlation (less than |0.1|) between the introduction of the new factor and the increase in retention at 7 and 30 days. However, the p-value = 0.611 >> 0.05 for Retention_7d is not statistically significant, while p-value = 0.037 < 0.05 for Retention_30d is statistically significan, but because the correlation is close to 0, this significance is practically meaningless. But this test only suggests the absence of a linear relationship between the introduction of a new feature and retention; further study of the data is necessary.

#### Task #2. Evaluate the impact of the feature using an RCT

Compare the mean values of Retention_7d and Retention_30d between the test and control groups.

Perform a t-test to check the statistical significance of the difference.

Can you say with certainty that the feature improves retention?

---

Separation into groups

In [9]:
test_group = df[df['Group'] == 'Test']
control_group = df[df['Group'] == 'Control']

Comparison of averages

In [10]:
mean_7d_test = test_group['Retention_7d'].mean()
mean_7d_control = control_group['Retention_7d'].mean()
mean_30d_test = test_group['Retention_30d'].mean()
mean_30d_control = control_group['Retention_30d'].mean()

print(f"Average Retention_7d - Test: {mean_7d_test:.3f}, Control: {mean_7d_control:.3f}")
print(f"Average Retention_30d - Test: {mean_30d_test:.3f}, Control: {mean_30d_control:.3f}")

Average Retention_7d - Test: 0.506, Control: 0.517
Average Retention_30d - Test: 0.465, Control: 0.511


T-test

In [11]:
tstat_7d, pval_7d = ttest_ind(test_group['Retention_7d'], control_group['Retention_7d'])
tstat_30d, pval_30d = ttest_ind(test_group['Retention_30d'], control_group['Retention_30d'])

print(f"T-test Retention_7d: t = {tstat_7d:.3f}, p = {pval_7d:.3f}")
print(f"T-test Retention_30d: t = {tstat_30d:.3f}, p = {pval_30d:.3f}")

T-test Retention_7d: t = -0.508, p = 0.611
T-test Retention_30d: t = -2.086, p = 0.037


#### Conclusion:

The difference between the mean values for the test and control groups over the 7-day period is very small. Small T-value = -0.508 means the difference between the means is tiny relative to variability in the data. p-value = 0.611 is again >> 0.05, so the difference is not statistically significant. We can conclude that there is no evidence that the test group’s 7-day retention differs from control.

The difference between mean values for the 30-day period is -4.6%, t-value = -2.1 shows a larger standardized difference, p-value = 0.037 < 0.05, so the difference is statistically significant. We can conclude that the test group’s 30-day retention is statistically significantly lower than control, even it's only -4.6%. Therefore, the introduction of a new feature had a negative effect.

#### Task #3: Using Propensity Score Matching (PSM)

Build a logistic regression that predicts the likelihood of using the feature based on Avg_Session_Time and Region.

Using the resulting propensity scores, match users from the test and control groups who have similar characteristics.

After matching, did the estimate of the feature’s impact on retention change?

---

Propensity score

In [12]:
# Coding categorical variables
df_encoded = pd.get_dummies(df, columns=["Region"], drop_first=True)

# Logistic regression
features = ['Avg_Session_Time'] + [col for col in df_encoded.columns if col.startswith("Region_")]
X = df_encoded[features]
y = df_encoded['Is_Test']

log_reg = LogisticRegression()
log_reg.fit(X, y)

# Adding the propensity score to the DataFrame
df_encoded['propensity_score'] = log_reg.predict_proba(X)[:, 1]

Matching with Nearest Neighbors

In [13]:
# Division into groups
tested = df_encoded[df_encoded['Is_Test'] == 1]
control = df_encoded[df_encoded['Is_Test'] == 0]

# Matching with k=1 neigbors
nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[['propensity_score']])
distances, indices = nn.kneighbors(tested[['propensity_score']])

# Extracting matched users
matched_control = control.iloc[indices.flatten()].copy()
matched_control.reset_index(drop=True, inplace=True)
matched_tested = tested.reset_index(drop=True)

# Comparison of retention after PSM
print("After the matching:")
print(f"Retention_7d - Test: {matched_tested['Retention_7d'].mean():.3f}, Control: {matched_control['Retention_7d'].mean():.3f}")
print(f"Retention_30d - Test: {matched_tested['Retention_30d'].mean():.3f}, Control: {matched_control['Retention_30d'].mean():.3f}")

# T-test on matched data
tstat_7d_psm, pval_7d_psm = ttest_ind(matched_tested['Retention_7d'], matched_control['Retention_7d'])
tstat_30d_psm, pval_30d_psm = ttest_ind(matched_tested['Retention_30d'], matched_control['Retention_30d'])

print(f"T-test Retention_7d (PSM): t = {tstat_7d_psm:.3f}, p = {pval_7d_psm:.3f}")
print(f"T-test Retention_30d (PSM): t = {tstat_30d_psm:.3f}, p = {pval_30d_psm:.3f}")

After the matching:
Retention_7d - Test: 0.506, Control: 0.577
Retention_30d - Test: 0.465, Control: 0.514
T-test Retention_7d (PSM): t = -3.212, p = 0.001
T-test Retention_30d (PSM): t = -2.221, p = 0.026


#### Conclusion:

The PSM methodology helps to find equivalent users between the test and control groups (in our case by region and average session time), as well as to check the correspondence between them.

For 7-day retention shows now -7.1% decrease, what is a bigger gap than it was found before, meaning the original overall average was masking a larger effect once you compare equivalent users. The t-value = -3.212, it's quite a large standardized difference. p-value << 0.05, so difference is statistically significant. Therefore, the test group has meaningfully lower 7-day retention than matched controls.

Regarding the 30-day retention, the difference is -4.6%, the t-value is large and p-value < 0.5. Therefore, the test group’s 30-day retention is also significantly lower.

Ultimately, it can be concluded that the proposed personalized recommendations feature reduces user engagement and should not be implemented.